# Unit 04 Worldle

Run this notebook top to bottom in JupyterLab. The last cell creates a playable Worldle-style game with an `ipyleaflet` mystery country map, a searchable country selector, guess history, a give-up button, and a new-game button.

## `wdo` work completed

- `wdo.geometry.bbox`: `bbox_from_feature`, `bbox_from_features`.
- `wdo.maps.leaflet_helpers`: `make_map`, `add_geojson`, `fit_map_to_geojson`, plus the small layer/path helpers needed by the notebook.
- `wdo.games.worldle`: `choose_target`, `feature_center`, `guess_feedback`, `format_feedback`, plus country metadata helpers and `build_country_lookup`.

## Data notes and aliases

- The assignment text mentions `countries_export.json`, but this checkout has `Resources/Data/countries.geojson`, so this notebook uses that file.
- This GeoJSON already has `ISO3166-1-Alpha-2`, so flags are joined primarily by ISO-2 instead of a fragile name-only join.
- The lookup still keeps name aliases for alternate data versions: `Czechia -> Czech Republic`, `Eswatini -> Swaziland`, and `North Macedonia -> Macedonia`.
- Taiwan uses `ISO3166-1-Alpha-2 = CN-TW` in the country polygons; the flag index uses `tw`, so `wdo.games.worldle` maps `CN-TW -> tw`.
- Some disputed or special territories use `-99` for ISO fields. Missing flags are shown as a small placeholder instead of breaking the row.

## Known bugs / honest limitations

- `feature_center(method="bbox")` is intentionally simple. Large antimeridian countries and island chains can produce imperfect centers and odd map lines.
- Guess direction and distance are based on representative country centers, not polygon-to-polygon nearest distance.
- Static notebook export cannot capture the interactive completed-round screenshot; run the notebook in Jupyter and play a round to inspect the final UI.

## Polish included

- Six-guess limit with target reveal on loss.
- Give-up reveal.
- Proximity color and temperature indicator in each guess row.
- Shareable result text after a win, loss, or give-up.
- Enter-to-guess support when the installed `ipywidgets` version exposes `Combobox.on_submit`.


In [1]:
from pathlib import Path
import base64
import html
import json
import sys

from IPython.display import display
import ipywidgets as widgets


def find_repo_root(start):
    '''Walk upward until the course Resources/wdo package is found.'''
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Resources" / "wdo" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError("Could not find Resources/wdo from this notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
RESOURCES = REPO_ROOT / "Resources"
if str(RESOURCES) not in sys.path:
    sys.path.insert(0, str(RESOURCES))

from wdo.games.worldle import (
    build_country_lookup,
    choose_target,
    country_iso2,
    country_iso3,
    country_name,
    format_feedback,
    guess_feedback,
)
from wdo.io.geojson_tools import iter_features, load_geojson
from wdo.maps.leaflet_helpers import (
    add_geojson,
    add_path,
    add_scale_control,
    fit_map_to_geojson,
    make_map,
)


In [2]:
DATA_DIR = REPO_ROOT / "Resources" / "Data"
COUNTRIES_PATH = DATA_DIR / "countries.geojson"
FLAG_INDEX_PATH = DATA_DIR / "flag-icons" / "country.json"
FLAG_BASE_DIR = DATA_DIR / "flag-icons"

countries_geojson = load_geojson(COUNTRIES_PATH)
country_features = sorted(list(iter_features(countries_geojson)), key=country_name)
flag_index = json.loads(FLAG_INDEX_PATH.read_text(encoding="utf-8"))
country_lookup = build_country_lookup(
    countries_geojson,
    flag_index,
    flag_base_dir=FLAG_BASE_DIR,
)


def feature_key(feature):
    '''Use ISO-3 when it is real; fall back to name for -99 territories.'''
    iso3 = country_iso3(feature)
    return iso3 if iso3 and iso3 != "-99" else country_name(feature)


def feature_label(feature):
    iso3 = country_iso3(feature)
    name = country_name(feature)
    if iso3 and iso3 != "-99":
        return f"{name} [{iso3}]"
    return name


def is_standard_feature(feature):
    iso2 = country_iso2(feature)
    iso3 = country_iso3(feature)
    return bool((iso3 and iso3 != "-99") or (iso2 and iso2 != "-99"))


standard_features = [feature for feature in country_features if is_standard_feature(feature)]

print(f"Loaded {len(country_features)} total features from {COUNTRIES_PATH.relative_to(REPO_ROOT)}")
print(f"Standard country mode uses {len(standard_features)} features with real ISO-2 or ISO-3 codes")
print(f"Flag lookup entries: {len(country_lookup)}")


Loaded 258 total features from Resources/Data/countries.geojson
Standard country mode uses 238 features with real ISO-2 or ISO-3 codes
Flag lookup entries: 258


In [3]:
SECRET_STYLE = {
    "color": "#111827",
    "fillColor": "#22c55e",
    "weight": 2,
    "fillOpacity": 0.55,
}

REVEAL_STYLE = {
    "color": "#b91c1c",
    "fillColor": "#f97316",
    "weight": 3,
    "fillOpacity": 0.35,
}


def single_feature_collection(feature, include_properties=False):
    properties = dict(feature.get("properties", {})) if include_properties else {}
    return {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": properties,
                "geometry": feature.get("geometry"),
            }
        ],
    }


def lookup_for_feature(feature):
    return country_lookup.get(feature_key(feature), {})


def flag_data_uri(flag_path):
    if not flag_path:
        return None
    path = Path(flag_path)
    if not path.exists():
        return None
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/svg+xml;base64,{encoded}"


def proximity(distance_km):
    if distance_km < 500:
        return ("#15803d", "hot")
    if distance_km < 2000:
        return ("#b45309", "warm")
    return ("#b91c1c", "cold")


def render_flag(flag_path, label):
    uri = flag_data_uri(flag_path)
    if uri:
        safe_label = html.escape(label)
        return f'<img src="{uri}" alt="{safe_label} flag" style="width:38px;height:28px;object-fit:cover;border:1px solid #d1d5db">'
    return '<span style="display:inline-flex;width:38px;height:28px;align-items:center;justify-content:center;border:1px solid #d1d5db;color:#6b7280;font-size:11px">no flag</span>'


def render_guess_row(feedback):
    name = feedback["guess_name"]
    meta = country_lookup.get(feedback.get("guess_iso3")) or country_lookup.get(name, {})
    distance = feedback["distance_km"]
    color, temperature = proximity(distance)
    flag = render_flag(meta.get("flag_path"), name)
    safe_name = html.escape(name)
    if feedback["correct"]:
        direction = "correct"
        distance_text = "0 km"
        row_color = "#15803d"
    else:
        direction = f'{feedback["arrow"]} {html.escape(feedback["compass"])}'
        distance_text = f'{distance:,.0f} km'
        row_color = color
    return f'''
    <div style="display:grid;grid-template-columns:46px 1fr 88px 96px 76px;gap:10px;align-items:center;padding:8px 10px;border-bottom:1px solid #e5e7eb;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif">
        <div>{flag}</div>
        <div style="font-weight:650;color:#111827">{safe_name}</div>
        <div style="font-size:22px;color:{row_color};font-weight:750;text-align:center">{direction}</div>
        <div style="font-variant-numeric:tabular-nums;text-align:right;color:#111827">{distance_text}</div>
        <div style="text-transform:uppercase;font-size:12px;font-weight:700;color:{color};text-align:right">{temperature}</div>
    </div>
    '''


def render_target_reveal(game, title):
    target_name = country_name(game.target)
    meta = lookup_for_feature(game.target)
    flag = render_flag(meta.get("flag_path"), target_name)
    safe_title = html.escape(title)
    safe_name = html.escape(target_name)
    return f'''
    <div style="display:flex;align-items:center;gap:12px;padding:12px 14px;border:1px solid #d1fae5;background:#ecfdf5;color:#064e3b;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif">
        {flag}
        <div><div style="font-weight:800">{safe_title}</div><div>The target was <b>{safe_name}</b>.</div></div>
    </div>
    '''


def shareable_result(game):
    status = "WIN" if game.won else "LOSS"
    lines = [f"Worldle {status} {len(game.guesses)}/{game.max_guesses}"]
    for feedback in game.guesses:
        if feedback["correct"]:
            lines.append("correct")
        else:
            _, temperature = proximity(feedback["distance_km"])
            lines.append(f'{feedback["arrow"]} {feedback["distance_km"]:,.0f} km {temperature}')
    lines.append(f"Target: {country_name(game.target)}")
    return "\n".join(lines)


In [4]:
class WorldleGame:
    '''Small inspectable state object for one Worldle round.'''

    def __init__(self, features, seed=None, max_guesses=6):
        self.features = list(features)
        self.feature_by_key = {feature_key(feature): feature for feature in self.features}
        self.target = choose_target(self.features, seed=seed)
        self.max_guesses = max_guesses
        self.guesses = []
        self.guessed_keys = set()
        self.finished = False
        self.won = False
        self.gave_up = False
        self.map = make_map(center=(20, 0), zoom=2, layout=widgets.Layout(height="520px", width="100%"))
        add_scale_control(self.map)
        self.target_layer = add_geojson(
            self.map,
            single_feature_collection(self.target, include_properties=False),
            name="Mystery country",
            style=SECRET_STYLE,
        )
        fit_map_to_geojson(self.map, single_feature_collection(self.target, include_properties=False))

    def submit_guess(self, key):
        if self.finished:
            return {"error": "This round is already finished."}
        if key not in self.feature_by_key:
            return {"error": "Pick a country from the list first."}
        if key in self.guessed_keys:
            return {"error": "You already guessed that country."}

        guess = self.feature_by_key[key]
        feedback = guess_feedback(guess, self.target)
        self.guessed_keys.add(key)
        self.guesses.append(feedback)

        if feedback["correct"]:
            self.finished = True
            self.won = True
        else:
            color, _ = proximity(feedback["distance_km"])
            add_path(
                self.map,
                [feedback["guess_center"], feedback["target_center"]],
                color=color,
                weight=2,
                opacity=0.65,
            )
            if len(self.guesses) >= self.max_guesses:
                self.finished = True

        return feedback

    def give_up_now(self):
        self.finished = True
        self.gave_up = True
        self.won = False

    def reveal_target_on_map(self):
        layer = add_geojson(
            self.map,
            single_feature_collection(self.target, include_properties=False),
            name="Revealed target",
            style=REVEAL_STYLE,
        )
        fit_map_to_geojson(self.map, single_feature_collection(self.target, include_properties=False))
        return layer


In [5]:
class WorldleApp:
    def __init__(self, standard, all_items, seed=2026, max_guesses=6):
        self.features_by_mode = {
            "standard": list(standard),
            "all": list(all_items),
        }
        self.seed_input = widgets.IntText(value=seed, description="Seed", layout=widgets.Layout(width="160px"))
        self.max_guesses = max_guesses
        self.mode = widgets.ToggleButtons(
            options=[("Standard", "standard"), ("All features", "all")],
            value="standard",
            description="Mode",
        )
        self.country_box = widgets.Combobox(
            placeholder="Type a country name",
            ensure_option=True,
            continuous_update=False,
            layout=widgets.Layout(width="360px"),
        )
        self.guess_button = widgets.Button(description="Guess", button_style="primary")
        self.give_up_button = widgets.Button(description="Give up", button_style="warning")
        self.new_button = widgets.Button(description="New game", icon="refresh")
        self.banner = widgets.HTML()
        self.history = widgets.HTML()
        self.share = widgets.Textarea(
            description="Share",
            disabled=False,
            layout=widgets.Layout(width="100%", height="110px"),
        )
        self.map_holder = widgets.VBox()
        self.controls = widgets.HBox(
            [self.country_box, self.guess_button, self.give_up_button, self.new_button],
            layout=widgets.Layout(align_items="center", gap="8px"),
        )
        self.settings = widgets.HBox(
            [self.mode, self.seed_input],
            layout=widgets.Layout(align_items="center", gap="12px"),
        )
        self.ui = widgets.VBox(
            [self.settings, self.map_holder, self.controls, self.banner, self.history, self.share],
            layout=widgets.Layout(gap="10px"),
        )

        self.guess_button.on_click(self.on_guess)
        self.give_up_button.on_click(self.on_give_up)
        self.new_button.on_click(self.on_new_game)
        self.mode.observe(self.on_mode_change, names="value")
        if hasattr(self.country_box, "on_submit"):
            self.country_box.on_submit(self.on_guess)

        self.new_game()

    def active_features(self):
        return self.features_by_mode[self.mode.value]

    def build_options(self):
        labels = []
        self.label_to_key = {}
        for feature in self.active_features():
            label = feature_label(feature)
            labels.append(label)
            self.label_to_key[label] = feature_key(feature)
        self.country_box.options = labels
        self.country_box.value = ""

    def new_game(self):
        self.build_options()
        self.game = WorldleGame(
            self.active_features(),
            seed=self.seed_input.value,
            max_guesses=self.max_guesses,
        )
        self.map_holder.children = [self.game.map]
        self.history.value = self.history_header()
        self.share.value = ""
        self.banner.value = self.status_message()
        self.set_finished(False)

    def history_header(self):
        return '''
        <div style="display:grid;grid-template-columns:46px 1fr 88px 96px 76px;gap:10px;align-items:center;padding:7px 10px;background:#f3f4f6;border:1px solid #e5e7eb;border-bottom:0;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;font-size:12px;font-weight:800;text-transform:uppercase;color:#374151">
            <div>Flag</div><div>Guess</div><div style="text-align:center">Direction</div><div style="text-align:right">Distance</div><div style="text-align:right">Heat</div>
        </div>
        '''

    def status_message(self):
        remaining = self.game.max_guesses - len(self.game.guesses)
        return f'''
        <div style="padding:10px 12px;background:#eff6ff;border:1px solid #bfdbfe;color:#1e3a8a;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif">
            Mystery country selected. {remaining} guesses remaining.
        </div>
        '''

    def set_finished(self, finished):
        self.guess_button.disabled = finished
        self.give_up_button.disabled = finished
        self.country_box.disabled = finished

    def refresh_history(self):
        rows = [self.history_header()]
        rows.extend(render_guess_row(feedback) for feedback in self.game.guesses)
        self.history.value = "".join(rows)

    def finish_round(self, title):
        self.game.reveal_target_on_map()
        self.banner.value = render_target_reveal(self.game, title)
        self.share.value = shareable_result(self.game)
        self.set_finished(True)

    def on_guess(self, _=None):
        key = self.label_to_key.get(self.country_box.value)
        feedback = self.game.submit_guess(key)
        if feedback and feedback.get("error"):
            self.banner.value = f'<div style="padding:10px 12px;background:#fef2f2;border:1px solid #fecaca;color:#991b1b">{html.escape(feedback["error"])}</div>'
            return

        self.refresh_history()
        self.country_box.value = ""
        if feedback["correct"]:
            self.finish_round("You got it.")
        elif self.game.finished:
            self.finish_round("Out of guesses.")
        else:
            self.banner.value = self.status_message()

    def on_give_up(self, _=None):
        self.game.give_up_now()
        self.finish_round("Round revealed.")

    def on_new_game(self, _=None):
        self.seed_input.value = self.seed_input.value + 1
        self.new_game()

    def on_mode_change(self, change):
        self.new_game()


In [6]:
# Lightweight non-interactive checks before the app appears.
usa = next(feature for feature in country_features if country_name(feature) == "United States of America")
france = next(feature for feature in country_features if country_name(feature) == "France")
check = guess_feedback(usa, france)
assert not check["correct"]
assert check["distance_km"] > 0
assert check["arrow"]
assert "toward the target" in format_feedback(check)

app = WorldleApp(standard_features, country_features, seed=2026, max_guesses=6)
display(app.ui)


/var/folders/kp/2bl6xl8n4939qvvlvq_fc8500000gn/T/ipykernel_6709/188022792.py:49: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.country_box.on_submit(self.on_guess)
